# 06 - Gold Pipeline (Lakeflow Declarative Pipeline)

Builds the Gold star schema on top of Silver: dimensions, facts, and the
business-ready summary tables every dashboard reads from.

Runs in the *same* Lakeflow pipeline as `05_silver_pipeline.ipynb` (both
are listed under one pipeline resource in `resources/pipelines.yml`), so
`dlt.read("some_silver_table")` just works without re-reading from the
catalog.

Business metric formulas (gross sales, net sales, profit, margin %) come
from `config/common.yaml` (`business_rules.measures`) so Gold and every
dashboard use the exact same formula — never redefined ad hoc here.

In [ ]:
import yaml
import dlt
from pyspark.sql import functions as F

CATALOG = spark.conf.get("catalog_name", "maven_test")

with open("config/common.yaml") as f:
    common_cfg = yaml.safe_load(f)
MEASURES = common_cfg["business_rules"]["measures"]

### Dimensions

In [ ]:
@dlt.table(name="dim_region", comment="Grain: one row per region_id.")
def dim_region():
    return dlt.read("regions_clean")


@dlt.table(name="dim_date", comment="Grain: one row per calendar date.")
def dim_date():
    return dlt.read("calendar_clean")


@dlt.table(name="dim_promotion", comment="Grain: one row per promotion_id.")
def dim_promotion():
    return dlt.read("promotions_clean")


@dlt.table(name="dim_product", comment="Grain: one row per product_id (no history requirement).")
def dim_product():
    return dlt.read("products_clean")


@dlt.table(name="dim_store", comment="Grain: one row per (store_id, SCD2 version).")
def dim_store():
    return dlt.read("stores_scd2").join(dlt.read("regions_clean"), "region_id", "left")


@dlt.table(
    name="dim_customer",
    comment="Grain: one row per (customer_id, version). PII columns are masked at "
            "query time via Unity Catalog column masks (sql/grants/04_column_masks.sql), "
            "not here -- Gold stores the real value.",
)
def dim_customer():
    return dlt.read("customers_scd2")

### Facts

In [ ]:
@dlt.table(
    name="fact_sales",
    comment="Grain: one row per transaction line (transaction_id). Never mutated by a return.",
)
def fact_sales():
    txn = dlt.read("transactions_clean")
    products = dlt.read("products_clean")
    # sales_region is denormalized onto the fact row so a Unity Catalog row
    # filter can attach directly to gold.fact_sales for regional-manager RLS.
    stores = dlt.read("stores_scd2").filter("__END_AT IS NULL").select("store_id", "region_id")
    regions = dlt.read("regions_clean").select("region_id", "sales_region")
    store_region = stores.join(regions, "region_id", "left").select("store_id", "sales_region")

    joined = (
        txn.join(products, "product_id", "left")
        .join(store_region, "store_id", "left")
        .withColumn("gross_sales", F.expr(MEASURES["gross_sales"]))
        .withColumn("discount_amount", F.expr(MEASURES["discount_amount"]))
        .withColumn("net_sales", F.expr(MEASURES["net_sales"]))
        .withColumn("cost", F.expr(MEASURES["cost"]))
        .withColumn("profit", F.expr(MEASURES["profit"]))
        .withColumn("margin_pct", F.when(F.col("net_sales") != 0, F.expr(MEASURES["margin_pct"])).otherwise(F.lit(None)))
    )
    return joined.select(
        "transaction_id", "transaction_date", "stock_date", "product_id", "customer_id",
        "store_id", "sales_region", "quantity", "payment_method", "promotion_id",
        "gross_sales", "discount_amount", "net_sales", "cost", "profit", "margin_pct",
    )


@dlt.table(name="fact_returns", comment="Grain: one row per return_id.")
def fact_returns():
    returns = dlt.read("returns_clean")
    products = dlt.read("products_clean")
    return (
        returns.join(products, "product_id", "left")
        .withColumn("return_value", F.col("quantity") * F.col("product_retail_price"))
        .select("return_id", "return_date", "product_id", "store_id", "quantity", "return_reason", "return_value")
    )


@dlt.table(name="fact_online_orders", comment="Grain: one row per order_id (streaming).")
def fact_online_orders():
    orders = dlt.read_stream("online_orders_clean")
    products = dlt.read("products_clean")
    return (
        orders.join(products, "product_id", "left")
        .withColumn("order_value", F.col("quantity") * F.col("product_retail_price"))
        .select(
            "order_id", "customer_id", "product_id", "store_id", "quantity",
            "payment_method", "channel_mode", "event_timestamp", "order_value",
        )
    )


@dlt.table(name="fact_inventory_movement", comment="Grain: one row per restock_event_id (streaming).")
def fact_inventory_movement():
    return dlt.read_stream("inventory_restock_clean")

### Business-ready summary tables

Dashboards read these, never `fact_sales` directly, so every dashboard
gets the same pre-aggregated numbers.

In [ ]:
@dlt.table(name="revenue_by_region", comment="Revenue, profit, margin by region and month.")
def revenue_by_region():
    sales = dlt.read("fact_sales")
    stores = dlt.read("dim_store").filter("__END_AT IS NULL")
    dates = dlt.read("dim_date")
    return (
        sales.join(stores, "store_id", "inner")
        .join(dates, sales.transaction_date == dates.date, "inner")
        .groupBy("sales_region", "year", "month")
        .agg(
            F.sum("net_sales").alias("net_sales"),
            F.sum("profit").alias("profit"),
            (F.sum("profit") / F.sum("net_sales")).alias("margin_pct"),
            F.countDistinct("transaction_id").alias("transaction_count"),
        )
    )


@dlt.table(name="profit_by_category", comment="Profit and margin by product category and month.")
def profit_by_category():
    sales = dlt.read("fact_sales")
    products = dlt.read("dim_product")
    dates = dlt.read("dim_date")
    return (
        sales.join(products, "product_id", "inner")
        .join(dates, sales.transaction_date == dates.date, "inner")
        .groupBy("product_category", "year", "month")
        .agg(
            F.sum("net_sales").alias("net_sales"),
            F.sum("profit").alias("profit"),
            (F.sum("profit") / F.sum("net_sales")).alias("margin_pct"),
        )
    )


@dlt.table(name="returns_pct_by_month", comment="Return rate % by month.")
def returns_pct_by_month():
    sales = dlt.read("fact_sales").withColumn("month_key", F.date_format("transaction_date", "yyyy-MM"))
    returns = dlt.read("fact_returns").withColumn("month_key", F.date_format("return_date", "yyyy-MM"))

    sold = sales.groupBy("month_key").agg(F.sum("quantity").alias("sold_quantity"))
    returned = returns.groupBy("month_key").agg(
        F.sum("quantity").alias("returned_quantity"),
        F.sum("return_value").alias("return_value"),
    )
    return (
        sold.join(returned, "month_key", "left")
        .fillna(0, subset=["returned_quantity", "return_value"])
        .withColumn("return_rate_pct", F.expr(MEASURES["return_rate_pct"]) * 100)
    )


@dlt.table(
    name="daily_sales_by_region",
    comment="Grain: one row per (sales_region, sales_date). Feeds the demand-forecasting notebook.",
)
def daily_sales_by_region():
    sales = dlt.read("fact_sales")
    stores = dlt.read("dim_store").filter("__END_AT IS NULL")
    return (
        sales.join(stores, "store_id", "inner")
        .groupBy("sales_region", "transaction_date")
        .agg(
            F.sum("net_sales").alias("net_sales"),
            F.sum("profit").alias("profit"),
            F.countDistinct("transaction_id").alias("transaction_count"),
        )
        .withColumnRenamed("transaction_date", "sales_date")
    )


@dlt.table(name="top_stores_products", comment="Top stores and products by net sales.")
def top_stores_products():
    sales = dlt.read("fact_sales")
    products = dlt.read("dim_product")
    stores = dlt.read("dim_store").filter("__END_AT IS NULL")
    by_store = (
        sales.join(stores, "store_id", "inner")
        .groupBy("store_id", "store_name")
        .agg(F.sum("net_sales").alias("net_sales"))
        .withColumn("rank_type", F.lit("store"))
    )
    by_product = (
        sales.join(products, "product_id", "inner")
        .groupBy("product_id", "product_name")
        .agg(F.sum("net_sales").alias("net_sales"))
        .withColumn("rank_type", F.lit("product"))
    )
    return by_store.select("rank_type", F.col("store_name").alias("name"), "net_sales").unionByName(
        by_product.select("rank_type", F.col("product_name").alias("name"), "net_sales")
    )